# Point Forecast – Alle Baseline Modelle
## Wind Power Forecasting | Seminar WIBA

Dieses Notebook implementiert alle Point Forecast Modelle und vergleicht sie wirtschaftlich.

| Modell | Typ |
|--------|-----|
| Persistence | Naiv – ŷ = ω_{t-24} |
| Mean Forecast | Naiv – ŷ = ω̄_train |
| Linear Regression | Interpretierbar |
| Random Forest | ML / semi-interpretierbar |
| Neural Network (MLP) | Black Box |

**Kernfrage:** Ist das beste RMSE-Modell auch das beste Trading-Modell?

---

## 1. Imports und Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

np.random.seed(42)
torch.manual_seed(42)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size']  = 11
sns.set_style('whitegrid')
print('Imports erfolgreich.')

## 2. Daten laden

Echte Winddaten – Schonungen 2016, stündlich aggregiert.

In [ ]:
DATA_PATH = '/Users/noahweis/wind_bidding_project/data/processed/final_dataset.csv'

df = pd.read_csv(DATA_PATH, parse_dates=['timestamp'])
df = df.sort_values('timestamp').reset_index(drop=True)

print(f'Datensatz: {df.shape[0]} Zeilen, {df.shape[1]} Spalten')
print(f'Zeitraum:  {df.timestamp.min()} bis {df.timestamp.max()}')
print(f'\nSpalten: {df.columns.tolist()}')
print(f'\nFehlende Werte:')
print(df.isnull().sum())
df.head()

### 2.1 Zielvariable visualisieren

In [ ]:
TARGET = 'power'

fig, axes = plt.subplots(2, 1, figsize=(14, 6))
axes[0].plot(df['timestamp'][:720], df[TARGET][:720], linewidth=0.8, color='#4C72B0')
axes[0].set_title('Windproduktion – erste 30 Tage')
axes[0].set_ylabel('Leistung [kW]')

axes[1].hist(df[TARGET], bins=60, color='#4C72B0', edgecolor='white', alpha=0.8)
axes[1].set_title('Verteilung der Windproduktion')
axes[1].set_xlabel('Leistung [kW]')
axes[1].set_ylabel('Häufigkeit')

plt.tight_layout()
plt.show()

print(f'Min:    {df[TARGET].min():.1f} kW')
print(f'Max:    {df[TARGET].max():.1f} kW')
print(f'Mean:   {df[TARGET].mean():.1f} kW')
print(f'Std:    {df[TARGET].std():.1f} kW')

## 3. Features und Train/Test Split (80/20)

`shuffle=False` – bei Zeitreihen keine zufällige Aufteilung!

In [ ]:
# Features automatisch erkennen
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
FEATURES = [c for c in numeric_cols if c != TARGET]
print(f'Target:   {TARGET}')
print(f'Features: {FEATURES}')

# 80/20 Split – zeitreihenkonform
TEST_SIZE = 0.2
split_idx  = int(len(df) * (1 - TEST_SIZE))

train_df = df.iloc[:split_idx].copy()
test_df  = df.iloc[split_idx:].copy()

X_train = train_df[FEATURES].values
X_test  = test_df[FEATURES].values
y_train = train_df[TARGET].values
y_test  = test_df[TARGET].values

# Skalierung (Pflicht für LR und NN)
scaler   = StandardScaler()
X_tr_sc  = scaler.fit_transform(X_train)
X_te_sc  = scaler.transform(X_test)

print(f'\nTrain: {len(X_train)} Samples ({100*(1-TEST_SIZE):.0f}%)')
print(f'Test:  {len(X_test)}  Samples ({100*TEST_SIZE:.0f}%)')

## 4. Hilfsfunktionen: Metriken

In [ ]:
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def mae(y_true, y_pred):
    return mean_absolute_error(y_true, y_pred)

def mape(y_true, y_pred, eps=1e-6):
    mask = np.abs(y_true) > eps
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

def newsvendor_loss(y_true, y_bid, c_over=10.0, c_under=15.0):
    """Asymmetrische Kostenfunktion – Kern des Bidding-Problems."""
    over  = np.maximum(y_bid - y_true, 0)
    under = np.maximum(y_true - y_bid, 0)
    return np.mean(c_over * over + c_under * under)

def evaluate(y_true, y_pred):
    return {
        'RMSE':            round(rmse(y_true, y_pred), 3),
        'MAE':             round(mae(y_true, y_pred), 3),
        'MAPE (%)':        round(mape(y_true, y_pred), 3),
        'Newsvendor Loss': round(newsvendor_loss(y_true, y_pred), 3),
    }

results     = {}  # Metriken je Modell
predictions = {}  # y_pred je Modell
print('Metriken definiert.')

## 5. Modell 1 – Persistence Baseline

$$\hat{\omega}_{t} = \omega_{t-24}$$

Kein Training nötig – gestern gleiche Stunde als Prognose.

In [ ]:
power_all     = df[TARGET].values
y_persistence = np.empty(len(y_test))

for i in range(len(y_test)):
    lag_idx = split_idx + i - 24
    y_persistence[i] = power_all[lag_idx] if lag_idx >= 0 else y_train.mean()

results['Persistence']     = evaluate(y_test, y_persistence)
predictions['Persistence'] = y_persistence

print('Persistence:')
for k, v in results['Persistence'].items(): print(f'  {k}: {v}')

## 6. Modell 2 – Mean Forecast Baseline

$$\hat{\omega}_{t} = \bar{\omega}_{train}$$

Konstanter Trainingsdurchschnitt – untere Grenze der Modellleistung.

In [ ]:
y_mean = np.full(len(y_test), y_train.mean())

results['Mean Forecast']     = evaluate(y_test, y_mean)
predictions['Mean Forecast'] = y_mean

print(f'Trainingsdurchschnitt: {y_train.mean():.2f} kW')
print('Mean Forecast:')
for k, v in results['Mean Forecast'].items(): print(f'  {k}: {v}')

## 7. Modell 3 – Linear Regression

$$\hat{\omega} = \beta_0 + \beta_1 x_1 + \ldots + \beta_p x_p$$

Interpretierbarste ML-Baseline. Mit polynomialen Features (Grad 2) für Windkurve.

In [ ]:
lr_pipeline = Pipeline([
    ('poly', PolynomialFeatures(degree=2, include_bias=False)),
    ('lr',   LinearRegression())
])
lr_pipeline.fit(X_tr_sc, y_train)
y_pred_lr = np.clip(lr_pipeline.predict(X_te_sc), 0, None)

results['Linear Regression']     = evaluate(y_test, y_pred_lr)
predictions['Linear Regression'] = y_pred_lr

print('Linear Regression:')
for k, v in results['Linear Regression'].items(): print(f'  {k}: {v}')

In [ ]:
# Koeffizienten visualisieren (ohne Polynomial-Expansion)
lr_simple = LinearRegression().fit(X_tr_sc, y_train)
coef_df   = pd.Series(lr_simple.coef_, index=FEATURES).sort_values(key=abs, ascending=True)

fig, ax = plt.subplots(figsize=(8, max(4, len(FEATURES) * 0.5)))
colors  = ['#C44E52' if c > 0 else '#4C72B0' for c in coef_df]
coef_df.plot(kind='barh', ax=ax, color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Linear Regression – Koeffizienten (skalierte Features)')
ax.set_xlabel('β')
plt.tight_layout()
plt.show()

## 8. Modell 4 – Random Forest

$$\hat{\omega} = \frac{1}{B}\sum_{b=1}^{B} T_b(x)$$

Ensemble aus 200 Entscheidungsbäumen. Nichtlinear, robust, semi-interpretierbar.

In [ ]:
rf_model = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)  # kein Scaler nötig
y_pred_rf = np.clip(rf_model.predict(X_test), 0, None)

results['Random Forest']     = evaluate(y_test, y_pred_rf)
predictions['Random Forest'] = y_pred_rf

print('Random Forest:')
for k, v in results['Random Forest'].items(): print(f'  {k}: {v}')

In [ ]:
# Feature Importance
importance = pd.Series(rf_model.feature_importances_, index=FEATURES).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, max(4, len(FEATURES) * 0.5)))
importance.plot(kind='barh', ax=ax, color='#55A868')
ax.set_title('Random Forest – Feature Importance')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

## 9. Modell 5 – Neural Network (MLP)

Architektur: Input → 64 → ReLU → 32 → ReLU → 1

**Wichtig:** Skalierte Features verwenden, sonst instabiles Training.

In [ ]:
class MLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64), nn.ReLU(),
            nn.Linear(64, 32),        nn.ReLU(),
            nn.Linear(32, 1)
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

X_t    = torch.FloatTensor(X_tr_sc)
y_t    = torch.FloatTensor(y_train)
loader = DataLoader(TensorDataset(X_t, y_t), batch_size=64, shuffle=True)

nn_model  = MLP(X_tr_sc.shape[1])
optimizer = torch.optim.Adam(nn_model.parameters(), lr=1e-3)
criterion = nn.MSELoss()

losses = []
for epoch in range(100):
    nn_model.train()
    ep_loss = 0
    for xb, yb in loader:
        optimizer.zero_grad()
        loss = criterion(nn_model(xb), yb)
        loss.backward()
        optimizer.step()
        ep_loss += loss.item()
    losses.append(ep_loss / len(loader))
    if (epoch + 1) % 20 == 0:
        print(f'Epoch {epoch+1:3d}/100 – Loss: {losses[-1]:.4f}')

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(losses, color='#C44E52')
ax.set_title('Neural Network – Trainingsverlust')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
plt.tight_layout()
plt.show()

In [ ]:
nn_model.eval()
with torch.no_grad():
    y_pred_nn = nn_model(torch.FloatTensor(X_te_sc)).numpy()
y_pred_nn = np.clip(y_pred_nn, 0, None)

results['Neural Network']     = evaluate(y_test, y_pred_nn)
predictions['Neural Network'] = y_pred_nn

print('Neural Network:')
for k, v in results['Neural Network'].items(): print(f'  {k}: {v}')

## 10. Modellvergleich

### 10.1 Metriken-Tabelle

In [ ]:
df_results = pd.DataFrame(results).T.astype(float).round(3)

print('── Point Forecast Vergleich ──────────────────────────────')
print(df_results.to_string())
print()
print('Bestes RMSE:            ', df_results['RMSE'].idxmin())
print('Bestes Newsvendor Loss: ', df_results['Newsvendor Loss'].idxmin())

df_results.to_csv('../results/tables/point_forecast_metrics.csv')

### 10.2 Balkendiagramm Metriken

In [ ]:
model_colors = ['#AAAAAA','#DDAA00','#4C72B0','#55A868','#C44E52']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

df_results['RMSE'].sort_values().plot(
    kind='barh', ax=axes[0], color=model_colors, edgecolor='white')
axes[0].set_title('RMSE (niedriger = besser)')
axes[0].set_xlabel('RMSE [kW]')

df_results['Newsvendor Loss'].sort_values().plot(
    kind='barh', ax=axes[1], color=model_colors, edgecolor='white')
axes[1].set_title('Newsvendor Loss (niedriger = besser)')
axes[1].set_xlabel('Loss')

plt.suptitle('Point Forecast – Modellvergleich', fontsize=13)
plt.tight_layout()
plt.savefig('../results/figures/pf_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

### 10.3 Forecast vs. Actual – alle Modelle (2 Wochen)

In [ ]:
N_PLOT = 336  # 2 Wochen
colors_map = {'Persistence':'#AAAAAA','Mean Forecast':'#DDAA00',
               'Linear Regression':'#4C72B0','Random Forest':'#55A868','Neural Network':'#C44E52'}

fig, axes = plt.subplots(len(predictions), 1, figsize=(14, 3*len(predictions)), sharex=True)
for ax, (name, y_pred) in zip(axes, predictions.items()):
    ax.plot(y_test[:N_PLOT], label='Actual', color='black', linewidth=1.2, alpha=0.9)
    ax.plot(y_pred[:N_PLOT], label=name,     color=colors_map[name], linewidth=1.2, linestyle='--')
    ax.set_ylabel('kW')
    ax.set_title(f"{name}  |  RMSE={results[name]['RMSE']:.1f}  NV-Loss={results[name]['Newsvendor Loss']:.1f}")
    ax.legend(loc='upper right', fontsize=8)
axes[-1].set_xlabel('Stunde (Testperiode)')
plt.suptitle('Forecast vs. Actual – erste 2 Wochen', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('../results/figures/pf_forecast_vs_actual.png', dpi=150, bbox_inches='tight')
plt.show()

### 10.4 Scatter: Actual vs. Predicted

In [ ]:
fig, axes = plt.subplots(1, len(predictions), figsize=(4*len(predictions), 4))
for ax, (name, y_pred) in zip(axes, predictions.items()):
    ax.scatter(y_test, y_pred, alpha=0.15, s=4, color=colors_map[name])
    lim = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
    ax.plot(lim, lim, 'r--', linewidth=1)
    ax.set_title(f"{name}\nRMSE={results[name]['RMSE']:.1f}", fontsize=9)
    ax.set_xlabel('Actual [kW]')
    ax.set_ylabel('Predicted [kW]')
plt.suptitle('Actual vs. Predicted', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig('../results/figures/pf_scatter_all.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Newsvendor-Bidding

Point Forecasts → direktes Gebot: **y_bid = ŷ**

Optimales Gebot wäre: y* = F⁻¹(τ) mit τ = c_under / (c_under + c_over)

→ Quantile Regression und Elastic Net liefern strukturell bessere Gebote.

In [ ]:
C_OVER  = 10.0  # EUR/MWh – TODO: aus echten Preisdaten
C_UNDER = 15.0  # EUR/MWh – TODO: aus echten Preisdaten
tau     = C_UNDER / (C_UNDER + C_OVER)

print(f'c_under={C_UNDER}, c_over={C_OVER} → tau={tau:.3f}')
print(f'Optimales Gebot = {tau*100:.0f}%-Quantil der Produktionsverteilung')
print()
print('── Newsvendor Loss Ranking ──────────────────────────────')
for name, y_pred in predictions.items():
    nv = newsvendor_loss(y_test, y_pred, C_OVER, C_UNDER)
    print(f'  {name:<22}: {nv:.3f}')

## 12. Zusammenfassung

### Key Insights
1. **Persistence** ist Industriestandard-Benchmark – muss von allen ML-Modellen geschlagen werden
2. **Mean Forecast** = absolute Untergrenze – jedes trainierte Modell sollte besser sein
3. **Linear Regression** interpretierbar, aber begrenzt durch Linearitätsannahme
4. **Random Forest** profitiert von Nichtlinearität der Windkurve (P ~ v³)
5. **Neural Network** flexibelst, aber Black Box
6. **Bestes RMSE ≠ bestes Newsvendor Loss** – zentrale Erkenntnis des Seminars

### Offene TODOs
- [ ] c_under / c_over aus echten DA- und reBAP-Preisen ableiten
- [ ] Vergleich mit Quantile Regression (06_elastic_net.ipynb)
- [ ] SHAP-Analyse für RF und NN (05_xai_analysis.ipynb)